# प्रतिमा निर्माण अनुप्रयोग तयार करणे (Azure OpenAI)

LLMs म्हणजे फक्त मजकूर निर्मितीपुरते मर्यादित नाही. तुम्ही मजकूर वर्णनांमधून प्रतिमा देखील तयार करू शकता. प्रतिमा हे एक माध्यम आहे जे MedTech, आर्किटेक्चर, पर्यटन, गेम डेव्हलपमेंट, मार्केटिंग आणि बरेच काही क्षेत्रात उपयोगी आहे. या धड्यात आपण Microsoft Foundry वरील आजचे **GPT Image** मॉडेल वापरून काम करतो.

## शिकण्याचे उद्दिष्टे

या धड्याच्या शेवटी तुम्ही सक्षम असाल:

- प्रतिमा निर्मिती काय आहे आणि ती कुठे उपयोगी आहे हे समजावून सांगणे.
- `gpt-image` मॉडेल कुटुंबाचा समजून घेणे आणि ते जुने DALL·E मॉडेल्सपासून कसे वेगळे आहे हे जाणून घेणे.
- प्रतिमा निर्माण अनुप्रयोग तयार करणे आणि प्रतिमा संपादित करणे.

## प्रतिमा निर्मिती म्हणजे काय?

प्रतिमा निर्मिती मॉडेल्स मजकूर प्रॉम्प्टवरून प्रतिमा तयार करतात. आधुनिक मॉडेल्स जसे की `gpt-image` प्रशिक्षणादरम्यान मजकूर आणि प्रतिमांमधील संबंध शिकतात, नंतर हळूहळू यादृच्छिक आवाज (random noise) प्रतिमेत रूपांतरित करतात जे तुमच्या प्रॉम्प्टशी जुळते.

प्रतिमा मॉडेल्सचे दोन प्रसिद्ध कुटुंबे आहेत:

- **`gpt-image` (OpenAI)** — सध्याचे उत्पादन जे या धड्यात वापरले जाते. ते मजकूर-ते-प्रतिमा निर्मिती आणि प्रतिमा संपादन (मुखवटा mask वापरून inpainting) यास समर्थन देते.
- **Midjourney** — स्वतःची सेवा व Discord-आधारित कार्यप्रवाह असलेले लोकप्रिय तृतीय-पक्ष मॉडेल.

> जुने OpenAI प्रतिमा मॉडेल्स — **DALL·E 2** आणि **DALL·E 3** — हे पारंपारिक आहेत. DALL·E 3 नवीन वापरासाठी उपलब्ध नाही, आणि `create_variation` सारखे फिचर्स फक्त DALL·E 2 मध्ये होते. नवीन अनुप्रयोगांसाठी `gpt-image` मॉडेल वापरा.

Microsoft Foundry वर, **`gpt-image-2`** हे सर्वात नवीन व सर्वाधिक सक्षम प्रतिमा मॉडेल आहे आणि शिफारसीय डीफॉल्ट आहे. `gpt-image-1.5` आणि `gpt-image-1-mini` देखील सामान्यतः उपलब्ध आहेत.

> **महत्त्वाचे:** `gpt-image` मॉडेल्स तयार केलेली प्रतिमा **base64** (`b64_json`) स्वरूपात परत करतात, URL स्वरूपात नाही. तुमच्या कोडद्वारे base64 स्ट्रिंग बाइट्समध्ये डीकोड करून ती जतन केली जाते — डाउनलोडसाठी कोणतीही प्रतिमा URL उपलब्ध नाही.


## तुमचे प्रथम प्रतिमा निर्माण अनुप्रयोग तयार करणे

तर प्रतिमा निर्माण अनुप्रयोग तयार करण्यासाठी काय आवश्यक आहे? तुम्हाला खालील लायब्ररींची गरज आहे:

- **python-dotenv**, हा लायब्ररी वापरण्याची तुम्हाला जोरदार शिफारस केली जाते जेणेकरून तुमचे रहस्य एका *.env* फाईलमध्ये कोडपासून वेगळे ठेवता येतील.
- **openai**, ही लायब्ररी तुम्ही OpenAI API सोबत संवाद साधण्यासाठी वापराल.
- **pillow**, Python मध्ये प्रतिमांसोबत काम करण्यासाठी.

जर आधी केले नसेल तर, [Microsoft Learn](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/create-resource?pivots=web-portal&WT.mc_id=academic-105485-koreyst) पृष्ठावरील सूचना अनुसरा आणि Microsoft Foundry संसाधन व मॉडेल तयार करा. मॉडेल म्हणून **gpt-image-2** निवडा (नवीनतम Azure OpenAI प्रतिमा मॉडेल; DALL·E हे जुनाट आहे).

1. *.env* नावाची एक फाईल तयार करा ज्यामध्ये खालील माहिती असावी:

    ```text
    AZURE_OPENAI_ENDPOINT=<your endpoint>
    AZURE_OPENAI_API_KEY=<your key>
    AZURE_OPENAI_DEPLOYMENT="gpt-image-2"
    ```

    ही माहिती Microsoft Foundry पोर्टलमध्ये तुमच्या संसाधनाच्या "Deployments" विभागात शोधू शकता.



1. वरील ग्रंथालये *requirements.txt* नावाच्या फाईलमध्ये खालीलप्रमाणे एकत्र करा:

    ```text
    python-dotenv
    openai
    pillow
    ```


1. नंतर, आभासी वातावरण तयार करा आणि ग्रंथालये स्थापीत करा:


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> विंडोज़साठी, आपले व्हर्च्युअल वातावरण तयार करण्यासाठी आणि सक्रिय करण्यासाठी खालील आदेश वापरा:

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. *app.py* नावाच्या फाईलमध्ये खालील कोड जोडा:

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    from openai import AzureOpenAI, BadRequestError

    # dotenv आयात करा
    dotenv.load_dotenv()

    # Azure OpenAI (Microsoft Foundry) क्लायंट कॉन्फिगर करा.
    # प्रतिमा मॉडेलसाठी नवीनतम API आवृत्ती आवश्यक आहे — तुमच्या मॉडेलला जे लागेल ते तपासण्यासाठी Foundry दस्तऐवज पहा.
    client = AzureOpenAI(
      azure_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"],
      api_key=os.environ['AZURE_OPENAI_API_KEY'],
      api_version = "2025-04-01-preview"
      )

    try:
        # प्रतिमा निर्माण API वापरून प्रतिमा तयार करा
        generation_response = client.images.generate(
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # तुमचा प्रॉम्प्ट मजकूर येथे टाका
            size='1024x1024',
            n=1,
            model=os.environ['AZURE_OPENAI_DEPLOYMENT']   # उदा. "gpt-image-2"
        )
        # साठवलेल्या प्रतिमेसाठी निर्देशिका सेट करा
        image_dir = os.path.join(os.curdir, 'images')

        # जर निर्देशिका अस्तित्वात नसेल तर ती तयार करा
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # प्रतिमेचा मार्ग प्रारंभ करा (फाईल प्रकार png असावा याची नोंद घ्या)
        image_path = os.path.join(image_dir, 'generated-image.png')

        # gpt-image मॉडेल प्रतिमा base64 (b64_json) स्वरूपात परत करतात, URL स्वरूपात नाही
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # डिफॉल्ट प्रतिमा दर्शकात प्रतिमा दर्शवा
        image = Image.open(image_path)
        image.show()

    # अपवाद पकडा
    except BadRequestError as err:
        print(err)

    ```

चला हा कोड समजावून घेऊया:

- प्रथम, आम्ही आवश्यक लायब्ररी आयात करतो, ज्यात OpenAI लायब्ररी, dotenv लायब्ररी, base64 मॉड्यूल, आणि Pillow लायब्ररी यांचा समावेश आहे.

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    from openai import AzureOpenAI, BadRequestError
    ```

- त्यानंतर, आम्ही *.env* फाईलमधून पर्यावरणातील परिवर्तने लोड करतो.

    ```python
    # dotenv आयात करा
    dotenv.load_dotenv()
    ```

- त्यानंतर, आम्ही Azure OpenAI (Microsoft Foundry) ग्राहकाचे कॉन्फिगर करतो.

    ```python
    client = AzureOpenAI(
      azure_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"],
      api_key=os.environ['AZURE_OPENAI_API_KEY'],
      api_version = "2025-04-01-preview"
      )
    ```

- पुढे, आम्ही प्रतिमा तयार करतो:

    ```python
    generation_response = client.images.generate(
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # आपला प्रॉम्प्ट मजकूर येथे लिहा
        size='1024x1024',
        n=1,
        model=os.environ['AZURE_OPENAI_DEPLOYMENT']
    )
    ```

    `gpt-image` मॉडेल प्रतिमा **base64** स्ट्रिंग म्हणून `data[0].b64_json` मध्ये परत करतं. आम्ही ते बाइट्समध्ये डीकोड करतो आणि फाईलमध्ये लिहितो — डाउनलोड करण्यासाठी कोणताही URL नाही.

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- शेवटी, आम्ही प्रतिमा उघडतो आणि मानक प्रतिमा दर्शक वापरून ती स्क्रीनवर दाखवतो:

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### प्रतिमा तयार करण्याविषयी अधिक तपशील

चला `images.generate` च्या पॅरामीटर्सवर नजर टाकूया:

- **prompt**, ही प्रतिमा तयार करण्यासाठी वापरलेली मजकूर संदर्भ आहे. येथे ती "धुके असलेल्या मेदानावर घेणारा घोड्यावर बसलेला बन्नी, ज्याच्याकडे लॉलीपॉप आहे, जिथे डेफोडिल्स वाढतात" आहे.
- **size**, तयार होणारी प्रतिमेचा आकार (उदा. `1024x1024`, `1536x1024`, `1024x1536`, किंवा `"auto"`).
- **n**, तयार होणाऱ्या प्रतिमांची संख्या. येथे आम्ही एक तयार करतो.
- **model**, आपला प्रतिमा मॉडेल डिप्लॉयमेंट नाव (उदा. `gpt-image-2`).

> प्रतिमा मॉडेल्स `temperature` पॅरामीटर घेत नाहीत — तो मजकूर निर्मितीसाठी वापरला जातो. विविधता मिळवण्यासाठी API पुन्हा कॉल करा; विविधता कमी करण्यासाठी आपला prompt अधिक विशिष्ट करा.

## प्रतिमा निर्माणाच्या अतिरिक्त क्षमतांचा परिचय

आपण बघितले कसे थोड्या पाइथाॅन कोडने प्रतिमा निर्माण करावी. `gpt-image` मॉडेल्स आधीच असलेल्या प्रतिमेत **संपादनही** करू शकतात. प्रतिमा, ऐच्छिक **मास्क** (ज्याने बदलायच्या भागाची चिन्हे लावतात), आणि एक prompt द्या, तर आपण प्रतिमेचा काही भाग बदलू शकता — उदाहरणार्थ, आपल्या बन्नीला टोपी घालू शकता.

```python
response = client.images.edit(
  model=os.environ['AZURE_OPENAI_DEPLOYMENT'],
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# संपादने देखील base64 म्हणून परत केली जातात
edited_image = base64.b64decode(response.data[0].b64_json)
```

मूळ प्रतिमेमध्ये फक्त ससा आहे; अंतिम प्रतिमेत सशावर टोपी आहे.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
हा दस्तऐवज AI भाषांतर सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) चा वापर करून अनुवादित केला आहे. जरी आम्ही अचूकतेसाठी प्रयत्न करतो, तरी कृपया लक्षात घ्या की स्वयंचलित भाषांतरांमध्ये त्रुटी किंवा अचूकतेची कमतरता असू शकते. मूळ दस्तऐवज त्याच्या मूळ भाषेत अधिकृत स्रोत मानला पाहिजे. महत्त्वाची माहिती असल्यास, व्यावसायिक मानवी भाषांतराची शिफारस केली जाते. या भाषांतराच्या वापरामुळे उद्भवणाऱ्या कोणत्याही गैरसमज किंवा चुकीच्या अर्थलावणीसाठी आम्ही जबाबदार नाही.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
